In [1]:
#pip install pandas pyarrow wget astropy

In [1]:
import os
import pandas as pd
import pyarrow.parquet as pq
import wget
from astropy.coordinates import SkyCoord
import astropy.units as u
import gc
import requests
import time

In [3]:
#Ruta a los ficheros parquet de DES DR2 Y6 GOLD
ruta_parquet = 'https://desdr-server.ncsa.illinois.edu/despublic/y6a2_files/y6_gold/'

#Lista de ficheros parquet
interfijo = ['0', '192', '2', '256', '257', '260', '261', '262', '263', '269', 
            '271', '272', '273', '274', '275', '276', '277', '278', '279', '280',
            '281', '282', '283', '284', '285', '286', '291', '292', '293', '294',
            '295', '296', '297', '298', '299', '300', '304', '305', '320', '321',
            '322', '323', '328', '329', '330', '331', '352', '354', '362', '517',
            '518', '519', '521', '522', '523', '524', '525', '526', '527', '528',
            '529', '530', '531', '532', '533', '534', '535', '536', '537', '538',
            '539', '540', '541', '542', '543', '544', '545', '546', '547', '548', 
            '549', '550', '551', '552', '553', '554', '555', '556', '557', '558',
            '559', '560', '561', '562', '563', '564', '565', '566', '567', '568',
            '569', '570', '571', '572', '573', '574', '575', '608', '610', '616',
            '618', '709', '710', '711', '713', '715', '716', '717', '718', '719',
            '720', '721', '722', '723', '724', '725', '726', '727', '728', '729',
            '730', '731', '732', '737', '740', '741', '743', '752', '767']

parquet_files = [f'Y6_GOLD_2_2-{i}-0000.parquet' for i in interfijo]

#Creamos una carpeta temporal para las descargas de los archivos parquet
temp_dir = "temp_parquet"
os.makedirs(temp_dir, exist_ok=True)

#Creamos una carpeta para guardar los resultados de los crossmatches
output_dir = "crossmatches"
os.makedirs(output_dir, exist_ok=True)

#Ruta al fichero de DES
des_file = '/mnt/c/Users/Leire/Downloads/des.csv'

#Cargamos el catálogo de DES
des = pd.read_csv(des_file)

#Definimos las columnas de RA y DEC de DES
ra_des = 'Astrogeo_DES_full_catalogue_matches_good_RA'
dec_des = 'Astrogeo_DES_full_catalogue_matches_good_DEC'

In [2]:
#Definimos una función que descarge los ficheros con reintentos y si no lo consigue los salta
def download_file(url, local_path, max_tries=8, chunk=1024*1024):

    for attempt in range(1, max_tries+1):
        try:
            print(f"Intentando descargar {url}: intento {attempt}/{max_tries}")
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(local_path, 'wb') as f:
                    for data in r.iter_content(chunk_size=chunk):
                        f.write(data)
            print(f"Descargado: {local_path}")
            return True
        except Exception as e:
            print(f"Error descargando: {e}")
            time.sleep(5)

    print(f"Fallo: no se pudo descargar {url}")
    return False

In [4]:
#Definimos una función para hacer los crossmatches
def crossmatch(df1, df2, ra1, dec1, ra2, dec2, radius_arcsec=1.0):
    
    coords1 = SkyCoord(ra=df1[ra1].values*u.deg, dec=df1[dec1].values*u.deg)
    coords2 = SkyCoord(ra=df2[ra2].values*u.deg, dec=df2[dec2].values*u.deg)
    
    idx, d2d, _ = coords1.match_to_catalog_sky(coords2)
    max_sep = 1.0*u.arcsec
    mask = d2d < max_sep
    
    matched_1 = df1[mask].reset_index(drop=True)
    matched_2 = df2.iloc[idx[mask]].reset_index(drop=True)

    crossmatched = pd.concat([matched_1, matched_2], axis=1)
    
    return crossmatched

In [ ]:
#Ahora hacemos el bucle que llame a la función, haga los crossmatches y los vaya guardando
#Lo vamos a hacer por partes porque ocupa mucha memoria y peta
for fname in parquet_files:

    output_file = os.path.join(output_dir, fname.replace(".parquet", "_crossmatch.csv"))

    #Si el fichero existe lo saltamos
    if os.path.exists(output_file):
        print(f"Saltando {fname}, ya procesado")
        continue
    
    #Descargamos el fichero
    url = ruta_parquet + fname
    local_path = os.path.join(temp_dir, fname)
    
    if not os.path.exists(local_path):
        ok = download_file(url, local_path)
        if not ok:
            print(f"No se pudo descargar {fname}, saltando.\n")
            continue

    #Leemos el parquet y si falla se salta
    try:
        parquet_file = pq.ParquetFile(local_path)
    except:
        print(f"Error leyendo {fname}, borrando y saltando")
        os.remove(local_path)
        continue

    #Guardamos matches parciales
    all_matches = []

    #Procesamos cada bloque interno del parquet y liberamos memoria
    for i in range(parquet_file.num_row_groups):
        print(f"Procesando {fname}, bloque {i+1}/{parquet_file.num_row_groups}")
        batch = parquet_file.read_row_group(i).to_pandas()[['RA', 'DEC', 'THETA_J2000', 'ERRTHETA_IMAGE', 'A_IMAGE', 'ERRA_IMAGE',
                                                           'B_IMAGE', 'ERRB_IMAGE', 'BDF_MAG_G', 'BDF_MAG_G_CORRECTED', 'BDF_MAG_ERR_G',
                                                           'BDF_MAG_I', 'BDF_MAG_I_CORRECTED', 'BDF_MAG_ERR_I',
                                                           'BDF_MAG_R', 'BDF_MAG_R_CORRECTED', 'BDF_MAG_ERR_R',
                                                           'BDF_MAG_Y', 'BDF_MAG_Y_CORRECTED', 'BDF_MAG_ERR_Y',
                                                           'BDF_MAG_Z', 'BDF_MAG_Z_CORRECTED', 'BDF_MAG_ERR_Z',
                                                           'BDF_T', 'BDF_T_ERR', 'BDF_T_RATIO', 'PSF_T',
                                                           'BDF_G_1', 'BDF_G_2', 'PSF_G_1', 'PSF_G_2']]
        matches = crossmatch(des, batch, ra_des, dec_des, 'RA', 'DEC')
        all_matches.append(matches)
        del batch
        gc.collect()

    #Unimos y guardamos el resultado del parquet completo
    result = pd.concat(all_matches, ignore_index=True)
    result.to_csv(output_file, index=False)
    print(f"Crossmatch guardado en {output_file} ({len(result)} coincidencias)")

    #Borramos el fichero parquet
    del result, all_matches
    os.remove(local_path)
    gc.collect()
    print(f"Archivo temporal {fname} eliminado\n")

100% [....................................................................] 3559587192 / 3559587192 Procesando Y6_GOLD_2_2-0-0000.parquet, bloque 1/2
